# Decision Tree with Scikit-Learn - Day 6

## Overfitting and Pruning

Using the algorithm described previously, we can train a decision tree that will perfectly classify training examples, assuming the examples are separable. However, if the dataset contains noise, this tree will overfit to the data and show poor test accuracy.

---

## What is Overfitting?

When a decision tree perfectly classifies all training examples but performs poorly on new data, it has overfit. This happens when the tree learns the noise in the training data rather than the underlying pattern.

**Example:** A dataset with a linear relation between feature x and label y, but with added noise. An unregularized decision tree will predict all training examples correctly but fail on new data.

---

## Regularization Methods to Limit Overfitting

Apply one or both of the following regularization criteria while training:

| Method | Description | Scikit-Learn Parameter |
|--------|-------------|----------------------|
| Maximum Depth | Prevent tree from growing past a certain depth | `max_depth` |
| Minimum Samples in Leaf | Leaf with less than certain number of examples will not be split | `min_samples_leaf` |
| Minimum Samples Split | Minimum samples required to split a node | `min_samples_split` |
| Maximum Features | Maximum features to consider for split | `max_features` |

---

## Pruning

Pruning is a post-training regularization technique that selectively removes certain branches by converting non-leaf nodes to leaves.

**How it works:**
1. Use a validation dataset
2. Test if removing a branch improves model quality on validation data
3. If yes, prune that branch (convert to leaf)

**Note:** Using a validation dataset reduces the number of examples available for initial training.

---

## Cost Complexity Pruning (CCP)

Scikit-Learn provides cost complexity pruning (also known as minimal cost-complexity pruning).

**Formula:** R_alpha(T) = R(T) + alpha * |T|

where:
- R(T) is the misclassification rate
- |T| is the number of leaves
- alpha is the complexity parameter

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
# Create noisy data
np.random.seed(42)
X = np.linspace(0, 10, 100).reshape(-1, 1)
y_true = X.flatten()
y = y_true + np.random.normal(0, 2, size=100)

# Split data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Train overfitted tree (no regularization)
overfit_tree = DecisionTreeRegressor()
overfit_tree.fit(X_train, y_train)

# Train regularized tree (with max_depth)
pruned_tree = DecisionTreeRegressor(max_depth=3)
pruned_tree.fit(X_train, y_train)

print(f"Overfit Tree - Train R2: {overfit_tree.score(X_train, y_train):.3f}")
print(f"Overfit Tree - Val R2: {overfit_tree.score(X_val, y_val):.3f}")
print(f"Pruned Tree - Train R2: {pruned_tree.score(X_train, y_train):.3f}")
print(f"Pruned Tree - Val R2: {pruned_tree.score(X_val, y_val):.3f}")

In [ ]:
# Visualize overfitting vs pruning
X_test = np.linspace(0, 10, 200).reshape(-1, 1)

plt.figure(figsize=(12, 6))
plt.scatter(X_train, y_train, alpha=0.5, label='Train Data', color='blue')
plt.scatter(X_val, y_val, alpha=0.5, label='Validation Data', color='orange')

# Overfit prediction
y_pred_overfit = overfit_tree.predict(X_test)
plt.plot(X_test, y_pred_overfit, label='Overfit (No Pruning)', color='red', linewidth=2)

# Pruned prediction
y_pred_pruned = pruned_tree.predict(X_test)
plt.plot(X_test, y_pred_pruned, label='Pruned (max_depth=3)', color='green', linewidth=2)

plt.xlabel('Feature X')
plt.ylabel('Target Y')
plt.title('Overfitting vs Pruning in Decision Trees')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Effect of different min_samples_leaf values
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

min_samples_options = [2, 5, 10]

for idx, min_samples in enumerate(min_samples_options):
    tree = DecisionTreeRegressor(min_samples_leaf=min_samples)
    tree.fit(X_train, y_train)
    y_pred = tree.predict(X_test)
    
    axes[idx].scatter(X_train, y_train, alpha=0.3, color='gray')
    axes[idx].plot(X_test, y_pred, color='red', linewidth=2)
    axes[idx].set_xlabel('Feature X')
    axes[idx].set_ylabel('Target Y')
    axes[idx].set_title(f'min_samples_leaf = {min_samples}')
    axes[idx].grid(alpha=0.3)

plt.suptitle('Effect of Minimum Samples per Leaf', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Cost Complexity Pruning
path = overfit_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

# Train trees with different alphas
trees = []
for alpha in ccp_alphas:
    tree = DecisionTreeRegressor(ccp_alpha=alpha)
    tree.fit(X_train, y_train)
    trees.append(tree)

# Find best alpha
train_scores = [tree.score(X_train, y_train) for tree in trees]
val_scores = [tree.score(X_val, y_val) for tree in trees]

best_alpha_idx = np.argmax(val_scores)
best_alpha = ccp_alphas[best_alpha_idx]

print(f"Best ccp_alpha: {best_alpha:.4f}")
print(f"Best validation R2: {val_scores[best_alpha_idx]:.3f}")

In [ ]:
# Visualize pruning effect
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(ccp_alphas[:-1], train_scores[:-1], label='Train', marker='o')
plt.plot(ccp_alphas[:-1], val_scores[:-1], label='Validation', marker='o')
plt.axvline(x=best_alpha, color='red', linestyle='--', label=f'Best alpha = {best_alpha:.4f}')
plt.xlabel('ccp_alpha')
plt.ylabel('R2 Score')
plt.title('Cost Complexity Pruning')
plt.legend()
plt.grid(alpha=0.3)

# Best pruned tree
plt.subplot(1, 2, 2)
best_tree = DecisionTreeRegressor(ccp_alpha=best_alpha)
best_tree.fit(X_train, y_train)
y_pred_best = best_tree.predict(X_test)

plt.scatter(X_train, y_train, alpha=0.3, color='gray')
plt.plot(X_test, y_pred_best, color='green', linewidth=2, label=f'Pruned (alpha={best_alpha:.4f})')
plt.xlabel('Feature X')
plt.ylabel('Target Y')
plt.title('Best Pruned Tree')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare tree sizes
print("\nTree Size Comparison:")
print("="*50)
print(f"Overfit Tree - Nodes: {overfit_tree.tree_.node_count}, Leaves: {overfit_tree.tree_.n_leaves}")
print(f"Pruned Tree (max_depth=3) - Nodes: {pruned_tree.tree_.node_count}, Leaves: {pruned_tree.tree_.n_leaves}")
print(f"Best Pruned Tree (ccp) - Nodes: {best_tree.tree_.node_count}, Leaves: {best_tree.tree_.n_leaves}")
print("="*50)

In [ ]:
# Day 6 Completed
print("\n" + "="*50)
print("DAY 6 COMPLETED")
print("="*50)
print("Topics covered:")
print("- Overfitting in Decision Trees")
print("- Regularization methods (max_depth, min_samples_leaf)")
print("- Cost Complexity Pruning (CCP)")
print("- Finding optimal ccp_alpha")
print("- Comparing overfit vs pruned trees")
print("="*50)"
print("\nNext: Day 7 - Feature Importance")